## Part 0: Tests (not part of the process)

In [ ]:
##### Just Testing Seleniuma and Getting the Table Data

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time

# 1) Create driver and open the page
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
driver.get("https://www.tennisabstract.com/cgi-bin/leaders.cgi")

# 2) Wait to ensure JavaScript is done loading
time.sleep(3)  # or use WebDriverWait for a more robust solution

# 3) Option A: Extract the entire HTML and parse with BeautifulSoup
from bs4 import BeautifulSoup

soup = BeautifulSoup(driver.page_source, "html.parser")
table = soup.find("table", {"class": "tablesorter", "id": "matches"})
if table:
    rows = table.find_all("tr")
    for row in rows:
        cells = [td.get_text(strip=True) for td in row.find_all(["th", "td"])]
        print(cells)

# 4) Option B: Use Selenium's element-finding
#    (Example: get all rows of the table with Selenium methods)
try:
    # The #matches table has <thead>, <tbody>, <tfoot>, etc.
    table_element = driver.find_element(By.ID, "matches")
    body_rows = table_element.find_elements(By.TAG_NAME, "tr")
    for row in body_rows:
        # Each cell is a <td> or <th>
        cells = row.find_elements(By.CSS_SELECTOR, "td, th")
        print([c.text for c in cells])
except:
    print("Could not locate the table with ID='matches'")

# 5) Close the browser once you’re done
driver.quit()


In [ ]:
#### Saving one table as a csv

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time

import pandas as pd
from bs4 import BeautifulSoup

# 1) Create driver and open the page
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
driver.get("https://www.tennisabstract.com/cgi-bin/leaders.cgi")

# 2) Wait (or use an explicit WebDriverWait)
time.sleep(3)

# 3) Parse with BeautifulSoup
soup = BeautifulSoup(driver.page_source, "html.parser")

# 4) Locate the table by id='matches'
table = soup.find("table", {"class": "tablesorter", "id": "matches"})

all_rows = []  # will become list of lists

if table:
    # each <tr> is a row
    rows = table.find_all("tr")
    for row in rows:
        # each cell is either <th> or <td>
        cells = [td.get_text(strip=True) for td in row.find_all(["th", "td"])]
        all_rows.append(cells)

# close the driver when done
driver.quit()

# 5) Convert list-of-lists into a DataFrame
#    - First row is column headers
headers = all_rows[0]              # e.g. ["Rk", "Player", "M", "M W-L", ...]
data_rows = all_rows[1:]           # the rest are data rows

df = pd.DataFrame(data_rows, columns=headers)

# OPTIONAL: Sometimes the last row is "Average" summary
# If you don't want it, you can drop it:
# df = df.iloc[:-1]

# 6) Display the first few lines
print(df.head())

# 7) Save to CSV
df.to_csv("tennis_leaders.csv", index=False)
print("Data saved to tennis_leaders.csv")

## Part 01: Scraping the Data

### Project Folder

In [1]:
from datetime import datetime
from pathlib import Path

# ── Project folder ─────────────────────────────────────────────────────
# Each scraping run saves to its own dated folder.
# To use a custom name, replace the line below with:
#   PROJECT_NAME = "my-custom-run"
PROJECT_NAME = datetime.now().strftime("data-%d-%b-%Y").lower()

BASE_DIR = Path(r"D:/coding/tennis-analytics") / PROJECT_NAME
BASE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project folder: {BASE_DIR}")

Project folder: D:\coding\tennis-analytics\data-24-mar-2026


### Downloading all tables for (4) for top 50

In [2]:
import time
import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

def get_table_as_dataframe(driver):
    soup = BeautifulSoup(driver.page_source, "html.parser")
    table = soup.find("table", {"class": "tablesorter", "id": "matches"})
    if not table:
        return pd.DataFrame()
    all_rows = []
    for row in table.find_all("tr"):
        c = [td.get_text(strip=True) for td in row.find_all(["th", "td"])]
        all_rows.append(c)
    if len(all_rows) < 2:
        return pd.DataFrame()
    return pd.DataFrame(all_rows[1:], columns=all_rows[0])

# 1) Open page
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
driver.get("https://www.tennisabstract.com/cgi-bin/leaders.cgi")
time.sleep(3)

# 2) Serve (default)
df_serve = get_table_as_dataframe(driver)
df_serve.to_csv(BASE_DIR / "tennis_leaders_serve.csv", index=False)
print("Saved tennis_leaders_serve.csv")

# 3) Return
try:
    driver.find_element(By.CSS_SELECTOR, "span.statsw.stattab").click()
    time.sleep(3)
    df_return = get_table_as_dataframe(driver)
    df_return.to_csv(BASE_DIR / "tennis_leaders_return.csv", index=False)
    print("Saved tennis_leaders_return.csv")
except:
    print("Could not find or click the Return link.")

# 4) Breaks
try:
    driver.find_element(By.CSS_SELECTOR, "span.statsl.stattab").click()
    time.sleep(3)
    df_breaks = get_table_as_dataframe(driver)
    df_breaks.to_csv(BASE_DIR / "tennis_leaders_breaks.csv", index=False)
    print("Saved tennis_leaders_breaks.csv")
except:
    print("Could not find or click the Breaks link.")

# 5) More
try:
    driver.find_element(By.CSS_SELECTOR, "span.statst.stattab").click()
    time.sleep(3)
    df_more = get_table_as_dataframe(driver)
    df_more.to_csv(BASE_DIR / "tennis_leaders_more.csv", index=False)
    print("Saved tennis_leaders_more.csv")
except:
    print("Could not find or click the More link.")

driver.quit()


Saved tennis_leaders_serve.csv
Saved tennis_leaders_return.csv
Saved tennis_leaders_breaks.csv
Saved tennis_leaders_more.csv


### Downloading all tables (4) for 51-100 range

In [3]:
import time
import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

def get_table_as_dataframe(driver):
    soup = BeautifulSoup(driver.page_source, "html.parser")
    table = soup.find("table", {"class": "tablesorter", "id": "matches"})
    if not table:
        return pd.DataFrame()
    all_rows = []
    for row in table.find_all("tr"):
        c = [td.get_text(strip=True) for td in row.find_all(["th", "td"])]
        all_rows.append(c)
    if len(all_rows) < 2:
        return pd.DataFrame()
    return pd.DataFrame(all_rows[1:], columns=all_rows[0])

# 1) Open 51-100 page
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
driver.get("https://www.tennisabstract.com/cgi-bin/leaders.cgi?players=51-100")
time.sleep(3)

# 2) Serve (default)
df_serve = get_table_as_dataframe(driver)
df_serve.to_csv(BASE_DIR / "tennis_leaders_51_100_serve.csv", index=False)
print("Saved tennis_leaders_51_100_serve.csv")

# 3) Return
try:
    driver.find_element(By.CSS_SELECTOR, "span.statsw.stattab").click()
    time.sleep(3)
    df_return = get_table_as_dataframe(driver)
    df_return.to_csv(BASE_DIR / "tennis_leaders_51_100_return.csv", index=False)
    print("Saved tennis_leaders_51_100_return.csv")
except:
    print("Could not find or click Return link")

# 4) Breaks
try:
    driver.find_element(By.CSS_SELECTOR, "span.statsl.stattab").click()
    time.sleep(3)
    df_breaks = get_table_as_dataframe(driver)
    df_breaks.to_csv(BASE_DIR / "tennis_leaders_51_100_breaks.csv", index=False)
    print("Saved tennis_leaders_51_100_breaks.csv")
except:
    print("Could not find or click Breaks link")

# 5) More
try:
    driver.find_element(By.CSS_SELECTOR, "span.statst.stattab").click()
    time.sleep(3)
    df_more = get_table_as_dataframe(driver)
    df_more.to_csv(BASE_DIR / "tennis_leaders_51_100_more.csv", index=False)
    print("Saved tennis_leaders_51_100_more.csv")
except:
    print("Could not find or click More link")

driver.quit()


Saved tennis_leaders_51_100_serve.csv
Saved tennis_leaders_51_100_return.csv
Saved tennis_leaders_51_100_breaks.csv
Saved tennis_leaders_51_100_more.csv


## Part 2: Cleaning the Data

### Using Pandas to Drop the Last Row

In [4]:
import glob
import pandas as pd

csv_files = glob.glob(str(BASE_DIR / "*.csv"))

for file in csv_files:
    df = pd.read_csv(file)
    df = df.iloc[:-1]  # drop last row (ATP summary/average row)
    df.to_csv(file, index=False)
    print(f"Dropped last row from: {file}")


Dropped last row from: D:\coding\tennis-analytics\data-24-mar-2026\tennis_leaders_51_100_breaks.csv
Dropped last row from: D:\coding\tennis-analytics\data-24-mar-2026\tennis_leaders_51_100_more.csv
Dropped last row from: D:\coding\tennis-analytics\data-24-mar-2026\tennis_leaders_51_100_return.csv
Dropped last row from: D:\coding\tennis-analytics\data-24-mar-2026\tennis_leaders_51_100_serve.csv
Dropped last row from: D:\coding\tennis-analytics\data-24-mar-2026\tennis_leaders_breaks.csv
Dropped last row from: D:\coding\tennis-analytics\data-24-mar-2026\tennis_leaders_more.csv
Dropped last row from: D:\coding\tennis-analytics\data-24-mar-2026\tennis_leaders_return.csv
Dropped last row from: D:\coding\tennis-analytics\data-24-mar-2026\tennis_leaders_serve.csv


### Merging the Datasets

#### Method 1

In [5]:
import pandas as pd

CATEGORIES = ["serve", "return", "breaks", "more"]

def files_for_category(cat):
    return {
        "top50":  BASE_DIR / f"tennis_leaders_{cat}.csv",
        "51_100": BASE_DIR / f"tennis_leaders_51_100_{cat}.csv",
    }

def merge_category(cat):
    paths = files_for_category(cat)
    missing = [name for name, p in paths.items() if not p.exists()]
    if missing:
        print(f"[{cat}] Missing files: {missing}")
    frames = []
    for name, p in paths.items():
        if p.exists():
            df = pd.read_csv(p)
            print(f"[{cat}] {name:7s}: {len(df):4d} rows from {p.name}")
            frames.append(df)
    if not frames:
        print(f"[{cat}] Skipped (no files found).")
        return
    merged = pd.concat(frames, ignore_index=True)
    out_path = BASE_DIR / f"leaders_men_{cat}_all.csv"
    merged.to_csv(out_path, index=False)
    print(f"[{cat}] wrote {len(merged)} rows to {out_path.name}")

for cat in CATEGORIES:
    merge_category(cat)


[serve] top50  :   50 rows from tennis_leaders_serve.csv
[serve] 51_100 :   50 rows from tennis_leaders_51_100_serve.csv
[serve] wrote 100 rows to leaders_men_serve_all.csv
[return] top50  :   50 rows from tennis_leaders_return.csv
[return] 51_100 :   50 rows from tennis_leaders_51_100_return.csv
[return] wrote 100 rows to leaders_men_return_all.csv
[breaks] top50  :   50 rows from tennis_leaders_breaks.csv
[breaks] 51_100 :   50 rows from tennis_leaders_51_100_breaks.csv
[breaks] wrote 100 rows to leaders_men_breaks_all.csv
[more] top50  :   50 rows from tennis_leaders_more.csv
[more] 51_100 :   50 rows from tennis_leaders_51_100_more.csv
[more] wrote 100 rows to leaders_men_more_all.csv


#### Method 2

In [6]:
import pandas as pd

categories = ["serve", "return", "breaks", "more"]

for cat in categories:
    top_50     = BASE_DIR / f"tennis_leaders_{cat}.csv"
    mid_51_100 = BASE_DIR / f"tennis_leaders_51_100_{cat}.csv"

    df_all = pd.concat(
        [pd.read_csv(top_50), pd.read_csv(mid_51_100)],
        ignore_index=True
    )

    out = BASE_DIR / f"{cat}_data.csv"
    df_all.to_csv(out, index=False)
    print(f"Created {out.name} with {len(df_all)} rows.")


Created serve_data.csv with 100 rows.
Created return_data.csv with 100 rows.
Created breaks_data.csv with 100 rows.
Created more_data.csv with 100 rows.


#### Comparando Method 1 e Method 2

In [7]:
import pandas as pd

pairs = [
    ("leaders_men_serve_all.csv",   "serve_data.csv",   "serve"),
    ("leaders_men_return_all.csv",  "return_data.csv",  "return"),
    ("leaders_men_breaks_all.csv",  "breaks_data.csv",  "breaks"),
    ("leaders_men_more_all.csv",    "more_data.csv",    "more"),
]

def read_csv_strict(p):
    return pd.read_csv(p, dtype=str, keep_default_na=False, na_filter=False)

def compare_pair(left_path, right_path, label):
    left  = read_csv_strict(left_path)
    right = read_csv_strict(right_path)

    if list(left.columns) == list(right.columns) and left.equals(right):
        print(f"[{label}] PERFECT match.")
        return True

    if set(left.columns) != set(right.columns):
        extra_l = set(left.columns) - set(right.columns)
        extra_r = set(right.columns) - set(left.columns)
        print(f"[{label}] Column mismatch. Only in left: {sorted(extra_l)}; only in right: {sorted(extra_r)}")
        return False
    right = right[left.columns]

    vc_left  = left.apply(tuple, axis=1).value_counts()
    vc_right = right.apply(tuple, axis=1).value_counts()
    only_in_left  = (vc_left  - vc_right).fillna(0)
    only_in_left  = only_in_left[only_in_left > 0].astype(int)
    only_in_right = (vc_right - vc_left).fillna(0)
    only_in_right = only_in_right[only_in_right > 0].astype(int)

    if only_in_left.empty and only_in_right.empty:
        print(f"[{label}] Match ignoring row order.")
        return True

    print(f"[{label}] MISMATCH: {len(only_in_left)} rows only in Method1, {len(only_in_right)} only in Method2")
    return False

all_ok = True
for l, r, name in pairs:
    ok = compare_pair(BASE_DIR / l, BASE_DIR / r, name)
    all_ok = all_ok and ok
print("All pairs match." if all_ok else "Some pairs differ.")


[serve] PERFECT match.
[return] PERFECT match.
[breaks] PERFECT match.
[more] PERFECT match.
All pairs match.


In [8]:
import pandas as pd

PAIRS = [
    ("leaders_men_serve_all.csv",   "serve_data.csv"),
    ("leaders_men_return_all.csv",  "return_data.csv"),
    ("leaders_men_breaks_all.csv",  "breaks_data.csv"),
    ("leaders_men_more_all.csv",    "more_data.csv"),
]

def normalize(df):
    out = df.copy()
    for c in out.columns:
        out[c] = out[c].astype(str).fillna("").str.replace(r"\s+", " ", regex=True).str.strip()
    return out

def compare_files(file_a, file_b):
    path_a, path_b = BASE_DIR / file_a, BASE_DIR / file_b
    df_a = pd.read_csv(path_a, dtype=str).fillna("")
    df_b = pd.read_csv(path_b, dtype=str).fillna("")

    cols_a, cols_b = set(df_a.columns), set(df_b.columns)
    if cols_a != cols_b:
        print(f"Column sets differ: only in A: {sorted(cols_a - cols_b)}, only in B: {sorted(cols_b - cols_a)}")

    cols = sorted(cols_a & cols_b)
    dfa = normalize(df_a[cols]).sort_values(by=cols).reset_index(drop=True)
    dfb = normalize(df_b[cols]).sort_values(by=cols).reset_index(drop=True)

    equal = (cols_a == cols_b) and dfa.equals(dfb)
    if equal:
        print(f"MATCH: {file_a} == {file_b} (rows={len(dfa)}, cols={len(cols)})")
        return True

    if len(df_a) != len(df_b):
        print(f"  Row count differs: {len(df_a)} vs {len(df_b)}")
    merged = dfa.merge(dfb, on=cols, how="outer", indicator=True)
    only_a = merged[merged["_merge"] == "left_only"]
    only_b = merged[merged["_merge"] == "right_only"]
    print(f"  Rows only in {file_a}: {len(only_a)}")
    print(f"  Rows only in {file_b}: {len(only_b)}")
    return False

all_ok = True
for a, b in PAIRS:
    ok = compare_files(a, b)
    all_ok = all_ok and ok
print("All pairs match exactly." if all_ok else "Some pairs differ.")


MATCH: leaders_men_serve_all.csv == serve_data.csv (rows=100, cols=19)
MATCH: leaders_men_return_all.csv == return_data.csv (rows=100, cols=14)
MATCH: leaders_men_breaks_all.csv == breaks_data.csv (rows=100, cols=19)
MATCH: leaders_men_more_all.csv == more_data.csv (rows=100, cols=19)
All pairs match exactly.
